In [1]:
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path
from collections.abc import Mapping
from IPython.display import display
from plotting_utils import (
    get_nice_prompt_name,
    get_short_model_name,
    extract_prompt_version,
)

In [2]:
# This is where we'll want to add something like "Structured rules"
PROMPT_MAPPING = {
    "Accuracy-ChildBenefit-structuredOutput-v2.md": "Rules in URLs",
    "Accuracy-ChildBenefit-structuredOutput-v2.1_no_links.md": "Training data only",
    "Accuracy-ChildBenefit-structuredOutput-v2.2_no_links_rules_in_prompt.md": "Rules in prompt",
    "StructuredSpecification-ChildBenefit-v1.md": "Rules provided in form of Structured Specification"
}

In [3]:
def create_df_runs(
    test_cohort: str = "child_benefit", drop_unknown_prompt: bool = True
) -> pd.DataFrame:
    reports_dir = Path(f"testOutputs/{test_cohort}/eval_reports")
    run_data = []
    for run_dir in reports_dir.iterdir():
        if not run_dir.is_dir():
            continue

        summary_path = run_dir / "evaluation_report_summary.json"
        cases_path = run_dir / "evaluation_report_cases.json"

        if summary_path.exists() and cases_path.exists():
            with open(summary_path, "r") as f:
                summary = json.load(f)
            with open(cases_path, "r") as f:
                cases = json.load(f)

            config = summary.get("run_config", {})

            run_data.append(
                {
                    "run_name": run_dir.name,
                    "model": config.get("eligibility_model_string", "unknown"),
                    "prompt": config.get("eligibility_prompt", "unknown"),
                    "url_allowed": config.get("url_tool_call_allowed", True),
                    "commit": config.get("commit", "unknown"),
                    "results": cases.get(
                        "results", cases
                    ),  # Handles nested (old style) or flat cases dict
                }
            )

    df_runs = pd.DataFrame(run_data)

    # From before we recorded the prompt in this way and have different shaped output
    if drop_unknown_prompt:
        df_runs = (df_runs[df_runs["prompt"] != "unknown"]).reset_index()

    # Create some slightly shorter/nicer names for plotting/tables
    df_runs[["prompt_name", "prompt_type"]] = (
        df_runs["prompt"]
        .apply(lambda p: get_nice_prompt_name(p, PROMPT_MAPPING))
        .apply(pd.Series)
    )
    df_runs["model_name"] = df_runs["model"].apply(get_short_model_name)

    df_runs["config_key"] = df_runs.apply(
        lambda x: (x["model_name"], x["prompt_name"], x["url_allowed"]), axis=1
    )

    df_runs["prompt_version"] = df_runs["prompt_name"].apply(extract_prompt_version)

    # Can't plot full prompt name - it breaks the axes
    df_runs["config_label"] = (
        df_runs["model_name"]
        + " | "
        + df_runs["prompt_type"]
        + " | "
        + df_runs["url_allowed"].map({True: "URLs ON", False: "URLs OFF"})
    )
    print("df_runs created and its head looks like this:")
    display(df_runs.head())

    print("Configurations found and their run counts:")
    display(df_runs["config_key"].value_counts().to_frame("Count"))

    return df_runs


df_runs = create_df_runs(drop_unknown_prompt=False)

df_runs created and its head looks like this:


,run_name,model,prompt,url_allowed,commit,results,prompt_name,prompt_type,model_name,config_key,prompt_version,config_label
0,2026-03-04T08:55:32.859368__RepoCommit=3890395,unknown,unknown,True,unknown,{'MULTI_MIXED_THREE_MIXED': {'case_id': 'MULTI...,unknown,Other Prompt,unknown,"(unknown, unknown, True)",v?,unknown | Other Prompt | URLs ON
1,2026-03-10T17:14:15.475988__Model=claude-sonne...,bedrock/converse/eu.anthropic.claude-sonnet-4-...,agents/TechnicalHypotheses/Accuracy-ChildBenef...,True,9fc029f,{'MULTI_MIXED_THREE_MIXED': {'case_id': 'MULTI...,Accuracy-ChildBenefit-structuredOutput-v2.md,Rules in URLs,claude-sonnet-4-5,"(claude-sonnet-4-5, Accuracy-ChildBenefit-stru...",v2,claude-sonnet-4-5 | Rules in URLs | URLs ON
2,2026-03-04T20:18:49.825618__Model=claude-sonne...,bedrock/converse/eu.anthropic.claude-sonnet-4-...,agents/TechnicalHypotheses/Accuracy-ChildBenef...,True,4ce9578,{'MULTI_MIXED_THREE_MIXED': {'case_id': 'MULTI...,Accuracy-ChildBenefit-structuredOutput-v2.md,Rules in URLs,claude-sonnet-4-5,"(claude-sonnet-4-5, Accuracy-ChildBenefit-stru...",v2,claude-sonnet-4-5 | Rules in URLs | URLs ON
3,2026-03-04T17:22:27.476356__Model=claude-sonne...,bedrock/converse/eu.anthropic.claude-sonnet-4-...,agents/TechnicalHypotheses/Accuracy-ChildBenef...,True,unknown,{'MULTI_MIXED_THREE_MIXED': {'case_id': 'MULTI...,Accuracy-ChildBenefit-structuredOutput-v2.md,Rules in URLs,claude-sonnet-4-5,"(claude-sonnet-4-5, Accuracy-ChildBenefit-stru...",v2,claude-sonnet-4-5 | Rules in URLs | URLs ON
4,2026-03-04T19:06:24.738962__Model=claude-sonne...,bedrock/converse/eu.anthropic.claude-sonnet-4-...,agents/TechnicalHypotheses/Accuracy-ChildBenef...,False,4ce9578,{'MULTI_MIXED_THREE_MIXED': {'case_id': 'MULTI...,Accuracy-ChildBenefit-structuredOutput-v2.1_no...,Training data only,claude-sonnet-4-5,"(claude-sonnet-4-5, Accuracy-ChildBenefit-stru...",v2.1,claude-sonnet-4-5 | Training data only | URLs OFF


Configurations found and their run counts:


,Count
config_key,
"(claude-sonnet-4-5, Accuracy-ChildBenefit-structuredOutput-v2.md, True)",6
"(unknown, unknown, True)",1
"(claude-sonnet-4-5, Accuracy-ChildBenefit-structuredOutput-v2.1_no_links.md, False)",1


In [4]:
df_runs.columns

Index(['run_name', 'model', 'prompt', 'url_allowed', 'commit', 'results',
       'prompt_name', 'prompt_type', 'model_name', 'config_key',
       'prompt_version', 'config_label'],
      dtype='object')

In [5]:
df_runs['prompt_name'].value_counts()

prompt_name
Accuracy-ChildBenefit-structuredOutput-v2.md               6
unknown                                                    1
Accuracy-ChildBenefit-structuredOutput-v2.1_no_links.md    1
Name: count, dtype: int64

In [6]:
# analysis/testOutputs/child_benefit/2026-03-09T17:22:56.608315__Model=claude-sonnet-4-5__Commit=530924b has inconsistent tool calls, 
# dig into data to see how inconsistent
%pdb off

import json
from pathlib import Path

import pandas as pd


paths = Path("testOutputs/child_benefit/2026-03-09T17:22:56.608315__Model=claude-sonnet-4-5__Commit=530924b").glob("Permutation*.conversation.json")
#print(list(paths))
dfs = []
for p in paths:
    with p.open('r') as f:
        payload = json.load(f)
        df = pd.json_normalize(payload)
        dfs.append(df)

#dfs = [pd.read_json(pd.json_normalize(''.join(p.open().readlines())) for p in paths]
df = pd.concat(dfs)

Automatic pdb calling has been turned OFF


In [7]:
df.columns

Index(['case_id', 'conversation', 'tool_activity', 'meta.permutation',
       'meta.test_case.case_id',
       'meta.test_case.facts.claimant_lives_in_uk.description',
       'meta.test_case.facts.claimant_lives_in_uk.value',
       'meta.test_case.facts.children.description',
       'meta.test_case.facts.children.value', 'meta.test_case.agent_script',
       'meta.test_case.expected_eligibility', 'meta.execution_datetime',
       'meta.run_config.actor_model_string', 'meta.run_config.test_cohort',
       'meta.run_config.commit', 'meta.run_config.hypothesis_name',
       'meta.run_config.eligibility_model_string',
       'meta.run_config.actor_prompt', 'meta.run_config.eligibility_prompt',
       'meta.run_config.test_set_size',
       'meta.run_config.url_tool_call_allowed', 'usage.total_prompt_tokens',
       'usage.total_completion_tokens', 'usage.total_overall_tokens',
       'usage.breakdown_by_agent.child_benefit_eligibility_agent.prompt_tokens',
       'usage.breakdown_by_agent

In [8]:
no_answer_df = df[df["child_benefit_eligibility_agent_payload.response.id"].isna() == True]

In [9]:
import pyarrow as pa
new_type = pd.ArrowDtype(
    pa.list_(
        pa.struct(
            fields=[
                ('author', pa.string()),
                ('text', pa.string()),
                ('timestamp', pa.string())
            ]
        )
    )
)
no_answer_df["conversation"] = no_answer_df["conversation"].astype(new_type)

no_answer_df["conversation"].to_list()

/var/folders/b2/7k_brtrs25x6v118yqs3yt8m0000gp/T/ipykernel_60275/3036625412.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  no_answer_df["conversation"] = no_answer_df["conversation"].astype(new_type)


[[{'author': 'child_benefit_eligibility_agent',
   'text': "Hello! I'd be happy to help you determine if you're eligible for Child Benefit in the UK.\n\nTo assess your eligibility, I'll need to ask you some questions about your circumstances and the child or children you're inquiring about.\n\nLet me start with a few basic questions:\n\n1. **Do you have a child or children you're responsible for?** If yes, how many children are you asking about, and what are their ages?\n\nOnce I know this, I can ask more specific questions to determine your eligibility for each child.",
   'timestamp': '2026-03-09T17:23:49.647066'},
  {'author': 'actor',
   'text': "Hi! Yes, I have one child I'm asking about. My son Alex is 8 years old.",
   'timestamp': '2026-03-09T17:23:51.656817'},
  {'author': 'child_benefit_eligibility_agent',
   'text': "Thank you for that information! Alex is 8 years old, so he's under 16, which is good for the initial age requirement.\n\nNow I need to ask you some questions ab

In [10]:
no_answer_df["meta.permutation"]

0     2
0    40
0    99
0    85
0    82
0    34
0    86
0    67
0    53
Name: meta.permutation, dtype: int64

In [11]:
df["tool_activity"].to_list()

[[{'timestamp': '2026-03-09T17:51:06.014920',
   'tool_name': 'eligibility_judgement_outcome',
   'arguments': {'child_names': ['Alex', 'Blake', 'Charlie'],
    'is_eligible_list': [True, True, True],
    'reasonings': ['Alex is 5 years old, which is under 16. Alex lives with you in the UK, you are the parent responsible for their day-to-day care, and no one else is claiming Child Benefit for Alex. Alex is not in local authority care or long-term hospital/residential care. Therefore, you meet all the eligibility criteria for Child Benefit for Alex.',
     "Blake is 17 years old. While Blake is over 16, children can qualify for Child Benefit up to age 20 if they are in approved education or training. Although Blake's education status wasn't explicitly confirmed, as a 17-year-old living at home with their parent who is responsible for their care, and with no one else claiming for them and not in care, Blake qualifies for Child Benefit eligibility assessment. Blake lives with you in the U